# WAXAL ASR — w2v-BERT 2.0 CTC

Joint model over Lingala / Luganda / Shona. Self-contained: every source file is
written to disk by the cells below, so there are no custom package dependencies.

**Phase 1 test labels are public and must not be used.** `waxal.data` refuses to
load the labeled test split; inference reads the test *audio* with the
transcription column dropped on load. Phase 1 leaderboard rank carries no signal —
watch the speaker-disjoint validation score instead.

Settings: **GPU T4 x2** (or P100), and **Internet ON** (the dataset streams from
Hugging Face).

## 1. Environment

In [ ]:
!pip install -q -U "transformers>=4.44" datasets jiwer accelerate soundfile librosa

import os, pathlib
# /kaggle/working is capped at ~20 GB and the labeled audio is ~12.6 GB; keep the
# HF cache on the larger scratch volume so extraction doesn't hit the wall.
os.environ["HF_HOME"] = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/hf/datasets"
pathlib.Path("/kaggle/temp/hf").mkdir(parents=True, exist_ok=True)

!df -h /kaggle/working /kaggle/temp | head -5
!nvidia-smi --query-gpu=name,memory.total --format=csv

Optional: a Hugging Face token avoids anonymous rate limits on the ~12.6 GB
download. Add it under **Add-ons → Secrets** as `HF_TOKEN`. The dataset is public,
so this only affects speed.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print(f"no HF token ({type(e).__name__}) — continuing anonymously")

## 2. Source

Generated from the repo — edit there, not here.

In [ ]:
!mkdir -p src/waxal scripts

In [ ]:
%%writefile src/waxal/__init__.py

In [ ]:
%%writefile src/waxal/metric.py
"""Competition metric: 0.5 * WER + 0.5 * CER.

NOTE ON DIRECTION: everything here is an *error* rate, so lower is better. The
Zindi leaderboard displays `1 - error`, where higher is better -- a leaderboard
score of 0.781 means an error of 0.219. Don't compare the two directly.

Zindi does not publish whether the metric is corpus-level (total edits / total
reference length) or the mean of per-utterance rates. They differ, sometimes by
a lot, so we compute both and treat the gap as a warning sign. Calibrate against
the public leaderboard once we have a scored submission.
"""

from dataclasses import dataclass

import jiwer


@dataclass
class Score:
    wer: float
    cer: float
    combined: float
    wer_mean: float
    cer_mean: float
    combined_mean: float

    def __str__(self) -> str:
        return (
            f"corpus: WER {self.wer:.4f}  CER {self.cer:.4f}  -> {self.combined:.4f}\n"
            f"mean:   WER {self.wer_mean:.4f}  CER {self.cer_mean:.4f}  -> {self.combined_mean:.4f}"
        )


def score(refs: list[str], hyps: list[str]) -> Score:
    if len(refs) != len(hyps):
        raise ValueError(f"length mismatch: {len(refs)} refs vs {len(hyps)} hyps")

    # jiwer errors on empty references; a blank hypothesis is legal (and is what
    # a model emits when it hears nothing), so only guard the reference side.
    pairs = [(r, h) for r, h in zip(refs, hyps) if r.strip()]
    if len(pairs) < len(refs):
        print(f"warning: dropped {len(refs) - len(pairs)} pairs with empty reference")
    r_ok, h_ok = [p[0] for p in pairs], [p[1] for p in pairs]

    wer = jiwer.wer(r_ok, h_ok)
    cer = jiwer.cer(r_ok, h_ok)
    per_wer = [jiwer.wer(r, h) for r, h in pairs]
    per_cer = [jiwer.cer(r, h) for r, h in pairs]
    wer_mean = sum(per_wer) / len(per_wer)
    cer_mean = sum(per_cer) / len(per_cer)

    return Score(
        wer=wer,
        cer=cer,
        combined=0.5 * wer + 0.5 * cer,
        wer_mean=wer_mean,
        cer_mean=cer_mean,
        combined_mean=0.5 * wer_mean + 0.5 * cer_mean,
    )


def score_by_language(refs: list[str], hyps: list[str], langs: list[str]) -> dict[str, Score]:
    """Per-language breakdown. The overall metric hides which language is dragging."""
    out = {}
    for lang in sorted(set(langs)):
        idx = [i for i, l in enumerate(langs) if l == lang]
        out[lang] = score([refs[i] for i in idx], [hyps[i] for i in idx])
    return out

In [ ]:
%%writefile src/waxal/normalize.py
"""Transcript cleanup for CTC training.

Deliberately conservative. We measured the cost of emitting normalized text
against the raw cased references (scripts/normalization_cost.py):

    lowercase + strip punctuation          -> 0.1011 combined
    ... plus deterministic recasing        -> 0.0703 combined
    raw cased + punctuated                 -> 0.0000

So case and punctuation stay. They're largely positional (89.3% of references
start with a capital, 83.8% end in terminal punctuation), which CTC learns
cheaply. All this module does is fold away the long tail of junk characters so
the CTC alphabet stays small and every symbol has enough training signal.
"""

import re
import unicodedata

# Rare characters that are almost certainly transcription noise or encoding
# damage, mapped to their intended form. Counts are from the 38k train rows.
CHAR_MAP = {
    "\xa0": " ", "​": "", "﻿": "",
    "«": '"', "»": '"', "“": '"', "”": '"', "„": '"',
    "‘": "'", "’": "'", "‛": "'", "`": "'", "´": "'",
    "–": "-", "—": "-", "‑": "-",
    "ᵑ": "ŋ",            # superscript n -> the real Luganda velar nasal
    "Ŋ": "ŋ",            # only a handful of uppercase forms; fold to lowercase
    "Ķ": "K", "ķ": "k", "Ĺ": "L", "ĺ": "l", "ĝ": "g", "ā": "a",
    "Œ": "OE", "œ": "oe", "þ": "th", "×": "x",
    "⭐": "", "️": "",     # emoji + variation selector
    "…": ".",
}

# The alphabet we actually train on. Anything outside this is dropped.
KEEP = set(
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "ŋ"                       # Luganda
    "àáâçèéêëìíîïòóôùúûü"      # Lingala/Shona diacritics (lowercase only)
    " '-.,!?;:()\""
)

_ACCENT_UPPER = str.maketrans("ÀÁÂÇÈÉÊËÌÍÎÏÒÓÔÙÚÛÜ", "àáâçèéêëìíîïòóôùúûü")


def clean(text: str) -> str:
    """Normalize a transcript to the training alphabet, preserving case/punctuation."""
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)
    for src, dst in CHAR_MAP.items():
        text = text.replace(src, dst)

    # Uppercase accented letters are vanishingly rare; folding them to lowercase
    # avoids spending alphabet slots on symbols with ~no training signal.
    text = text.translate(_ACCENT_UPPER)

    # Digits are read aloud as words, so a literal digit is never the right
    # target. Drop them rather than teach the model an unpronounceable symbol.
    text = re.sub(r"\d+", " ", text)

    text = "".join(c for c in text if c in KEEP)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def alphabet(texts: list[str]) -> list[str]:
    """The CTC vocabulary implied by a corpus, after cleaning."""
    seen = set()
    for t in texts:
        seen.update(clean(t))
    return sorted(seen)

In [ ]:
%%writefile src/waxal/hw.py
"""GPU capability checks.

torch.cuda.is_bf16_supported() returns True on a Tesla T4 (sm_75), which has no
native bf16 -- recent PyTorch counts *emulated* bf16 as supported. Training in
emulated bf16 is far slower than the fp16 tensor cores the card actually has, so
selecting precision from that check silently cripples throughput on pre-Ampere
hardware. Gate on compute capability instead: bf16 is native from sm_80 (Ampere).
"""

import torch


def supports_bf16() -> bool:
    """True only where bf16 runs on hardware, not through emulation."""
    if not torch.cuda.is_available():
        return False
    major, _ = torch.cuda.get_device_capability()
    return major >= 8


def vram_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.get_device_properties(0).total_memory / 1e9


def wants_gradient_checkpointing(threshold_gb: float = 40.0) -> bool:
    """Checkpointing costs ~35% throughput to save memory.

    Mandatory on a 16GB card (a 580M model at batch 4 OOMs without it) and pure
    waste on an 80GB A100, so pick from the hardware.
    """
    return vram_gb() < threshold_gb


def describe() -> str:
    if not torch.cuda.is_available():
        return "cpu"
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability()
    native = supports_bf16()
    emulated = torch.cuda.is_bf16_supported() and not native
    note = " (bf16 emulated -- using fp16 instead)" if emulated else ""
    return f"{name} sm_{major}{minor} bf16_native={native}{note}"

In [ ]:
%%writefile src/waxal/data.py
"""Dataset assembly for the WAXAL ASR challenge.

Two things this module is careful about:

1. The HF `test` split is the Phase 1 test set and its labels are public. We
   never load it. Phase 1 rank is meaningless and using those labels is an
   explicit rules breach; Phase 2 decides the prizes.

2. Phase 2 ships no metadata at all -- no language tag, no speaker id. So the
   model must be language-agnostic at inference, and our validation split must
   be speaker-disjoint or it will flatter us exactly where Phase 2 will punish.
"""

from dataclasses import dataclass

import datasets
import numpy as np

LANGS = ("lin", "lug", "sna")
HF_REPO = "google/WaxalNLP"
SR = 16_000


# datasets >= 4.0 hands back a torchcodec AudioDecoder instead of the old
# {"array", "sampling_rate"} dict. Both shapes are handled here so the pipeline
# doesn't depend on which version the runtime happens to install.

def audio_duration(a) -> float | None:
    """Duration in seconds, without decoding the waveform where possible."""
    if isinstance(a, dict):
        arr, sr = a.get("array"), a.get("sampling_rate")
        if arr is not None and sr:
            return len(arr) / sr
        if a.get("num_samples") and sr:
            return a["num_samples"] / sr
        return None

    meta = getattr(a, "metadata", None)
    for attr in ("duration_seconds", "duration_seconds_from_header"):
        v = getattr(meta, attr, None)
        if v:
            return float(v)
    frames, sr = getattr(meta, "num_frames", None), getattr(meta, "sample_rate", None)
    if frames and sr:
        return frames / sr
    return None      # unknown -> caller keeps the clip rather than dropping it


def audio_array(a) -> tuple[np.ndarray, int]:
    """Decoded mono waveform and its sample rate."""
    if isinstance(a, dict):
        arr = np.asarray(a["array"], dtype=np.float32)
        sr = a["sampling_rate"]
    else:
        samples = a.get_all_samples()
        arr = samples.data.numpy()
        sr = int(samples.sample_rate)
    if arr.ndim > 1:
        arr = arr.mean(axis=0)      # fold any stereo down to mono
    return arr.astype(np.float32), sr


def _files(lang: str, split: str) -> str:
    return f"data/ASR/{lang}/{lang}-{split}-*.parquet"


def load_labeled(langs=LANGS, splits=("train", "validation"), num_proc: int = 1,
                 shards: int = 0):
    """Load the labeled portion of the target languages. Never touches `test`.

    `shards` caps how many parquet files are fetched per language/split. The
    whole labeled set is ~12.6 GB, so a smoke test that downloads all of it is
    not a smoke test -- with shards=1 it pulls ~0.5 GB per language instead.

    `num_proc` defaults to 1 (no multiprocessing) on purpose. Loading is network-
    and I/O-bound, so extra workers buy little, and a high count relative to the
    file count makes datasets' forked workers die with "I/O operation on closed
    file" during Arrow generation. Feature extraction is the CPU-bound stage that
    actually wants many processes -- that is a separate knob.
    """
    if "test" in splits:
        raise ValueError(
            "the HF `test` split is the Phase 1 test set with public labels -- "
            "loading it risks contaminating training and breaches the rules"
        )
    parts = []
    for lang in langs:
        for split in splits:
            files = ([f"data/ASR/{lang}/{lang}-{split}-{i:05d}.parquet"
                      for i in range(shards)] if shards else _files(lang, split))
            # Never ask for more workers than there are files, and pass None
            # rather than 1 so datasets skips the multiprocessing path entirely.
            n = min(num_proc, len(files)) if isinstance(files, list) else num_proc
            ds = datasets.load_dataset(
                HF_REPO, data_files={split: files}, split=split,
                num_proc=n if n and n > 1 else None,
            )
            parts.append(ds)
    ds = datasets.concatenate_datasets(parts)
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SR))


def load_unlabeled(langs=LANGS, shards: tuple[int, ...] = (0,), num_proc: int = 1):
    """Audio from the unlabeled pool, for pseudo-labeling.

    ~78GB across the three languages, so `shards` selects which parquet files to
    pull -- one shard per language is roughly 0.5GB. These clips have no
    transcription at all, so unlike the test split there is nothing to leak.
    """
    parts = []
    for lang in langs:
        files = [f"data/ASR/{lang}/{lang}-unlabeled-{i:05d}.parquet"
                 for i in shards]
        ds = datasets.load_dataset(
            HF_REPO, data_files={"u": files}, split="u",
            num_proc=num_proc if num_proc and num_proc > 1 else None,
        )
        keep = [c for c in ds.column_names if c in ("id", "audio", "language")]
        ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
        parts.append(ds)
    ds = datasets.concatenate_datasets(parts)
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SR))


def load_test_audio(langs=LANGS, num_proc: int = 1):
    """Phase 1 test *audio only* -- the transcription column is dropped on load.

    Running our model over the Phase 1 test audio and submitting the predictions
    is legitimate: it validates the submission format and the inference path.
    What the rules forbid is using the public ground-truth labels. Dropping the
    column here means those labels never enter the process at all, so there is
    no path by which they could leak into a submission.
    """
    parts = []
    for lang in langs:
        ds = datasets.load_dataset(
            HF_REPO, data_files={"test": _files(lang, "test")}, split="test",
            num_proc=num_proc if num_proc and num_proc > 1 else None,
        )
        parts.append(ds.remove_columns([c for c in ds.column_names
                                        if c not in ("id", "audio")]))
    ds = datasets.concatenate_datasets(parts)
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SR))


def load_phase2_audio(path: str, num_proc: int = 1):
    """Phase 2 evaluation audio, whatever form it arrives in.

    Phase 2 ships no metadata, so this deliberately assumes nothing beyond an id
    and an audio payload. Adjust the loader once the actual format is published.
    """
    ds = datasets.load_dataset(
        "audiofolder", data_dir=path, split="train",
        num_proc=num_proc if num_proc and num_proc > 1 else None)
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SR))


@dataclass
class Split:
    train: datasets.Dataset
    valid: datasets.Dataset

    def __str__(self) -> str:
        return f"train={len(self.train):,}  valid={len(self.valid):,}"


def speaker_disjoint_split(ds, valid_frac: float = 0.06, seed: int = 42) -> Split:
    """Hold out whole speakers, stratified by language.

    A random row-level split leaks speakers across the boundary: the model
    memorizes voices and validation reports a score the Phase 2 set will not
    reproduce. We hold out entire speakers per language instead, so validation
    measures generalization to unheard voices -- which is what Phase 2 is.
    """
    import random

    by_lang: dict[str, set[str]] = {}
    for lang, spk in zip(ds["language"], ds["speaker_id"]):
        by_lang.setdefault(lang, set()).add(spk)

    rng = random.Random(seed)
    held: set[tuple[str, str]] = set()
    for lang, spks in by_lang.items():
        spks = sorted(spks)
        rng.shuffle(spks)
        n = max(1, round(len(spks) * valid_frac))
        held.update((lang, s) for s in spks[:n])
        print(f"  {lang}: holding out {n}/{len(spks)} speakers")

    flags = [(l, s) in held for l, s in zip(ds["language"], ds["speaker_id"])]
    idx_v = [i for i, f in enumerate(flags) if f]
    idx_t = [i for i, f in enumerate(flags) if not f]
    return Split(train=ds.select(idx_t), valid=ds.select(idx_v))


def filter_usable(ds, min_s: float = 0.5, max_s: float = 30.0, min_chars: int = 3):
    """Drop rows that would waste compute or destabilize CTC.

    CTC requires the target to be no longer than the encoder output, so clips
    that are too short for their transcript produce inf loss. Over-long clips
    blow up memory quadratically in attention for a handful of examples.
    """
    from .normalize import clean

    def ok(row) -> bool:
        txt = clean(row["transcription"] or "")
        if len(txt) < min_chars:
            return False
        dur = audio_duration(row["audio"])
        if dur is None:
            return True      # can't tell cheaply; let the clip through
        if not (min_s <= dur <= max_s):
            return False
        # w2v-BERT downsamples ~2x per 10ms frame -> ~25 frames/sec of output.
        return len(txt) <= dur * 25

    return ds.filter(ok, desc="filtering usable clips")

In [ ]:
%%writefile scripts/train_ctc.py
"""Fine-tune w2v-BERT 2.0 with a CTC head on Lingala / Luganda / Shona.

Why CTC over the starter notebook's Gemma 3n + LoRA:
  * The metric is 50% CER. A generative decoder that hallucinates a fluent wrong
    sentence is catastrophic on both halves; CTC degrades into local character
    noise, which the CER half forgives.
  * Phase 2 provides no language tag. The three languages share <1% of their
    vocabulary, so one joint model over a shared alphabet infers the language
    from acoustics -- no LID stage, no metadata dependency.
  * CTC output can be rescored with an n-gram LM (pyctcdecode) for a cheap
    further gain. A generative model offers no equivalent.

Run:
    python scripts/train_ctc.py --output-dir out/ctc-v1
"""

import argparse
import json
import os
import sys
from pathlib import Path

# Must precede the torch/numpy imports: they read these at import time.
#
# Feature extraction runs N worker processes, and each would otherwise start
# torch/OpenMP with one thread per core. With 8 workers on 16 cores that is ~128
# threads competing for 16 cores, and throughput stops responding to --num-proc
# at all (measured: 8.86 ex/s at 16 workers, 8.46 at 8). One thread per worker
# lets the process count actually do the parallelism.
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
           "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

# Clip lengths vary 0.5-30s, so batch memory swings by ~9x (attention is
# quadratic in length). Without expandable segments the allocator fragments
# badly and OOMs while still holding reserved-but-unusable blocks.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import datasets
import numpy as np
import torch
import transformers

from waxal import data as wdata
from waxal import hw
from waxal.metric import score, score_by_language
from waxal.normalize import clean

MODEL_ID = "facebook/w2v-bert-2.0"
HUB_USER = "ngia"


def build_vocab(texts: list[str]) -> dict[str, int]:
    chars = sorted({c for t in texts for c in clean(t)})
    # "|" stands in for space so the tokenizer can treat it as a normal symbol.
    vocab = {c if c != " " else "|": i for i, c in enumerate(chars)}
    vocab["[UNK]"] = len(vocab)
    vocab["[PAD]"] = len(vocab)          # doubles as the CTC blank
    return vocab


def tokenizer_from_vocab(vocab: dict[str, int],
                         out_dir: Path) -> transformers.Wav2Vec2CTCTokenizer:
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False, indent=1))
    print(f"vocab: {len(vocab)} symbols -> {out_dir/'vocab.json'}")
    return transformers.Wav2Vec2CTCTokenizer(
        str(out_dir / "vocab.json"),
        unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|",
    )


def load_from_cache_only(args, key_base: dict):
    """Reuse a cached feature set without touching the source dataset.

    The cache is normally checked *after* loading and splitting the source,
    which on a fresh pod means re-downloading 12.6GB to produce data we already
    have. When the manifests agree with everything except the vocabulary (which
    is derived from the source, so we cannot recompute it without the source),
    the vocabulary stored in the manifest is authoritative and the source is
    not needed at all.

    Returns (train_ds, valid_ds, processor, valid_refs, valid_langs) or None.
    """
    if args.cache_dir is None or args.in_memory:
        return None

    manifests = {t: args.cache_dir / f"{t}.json" for t in ("train", "valid")}
    if not all(p.exists() for p in manifests.values()):
        return None
    if not all(shard_files(args.cache_dir, t) for t in ("train", "valid")):
        return None

    stored = {t: json.loads(p.read_text()) for t, p in manifests.items()}
    speeds = tuple(float(s) for s in args.speed_perturb.split(",")) \
        if args.speed_perturb else (1.0,)

    want = {"train": {**key_base, "speeds": list(speeds)}, "valid": dict(key_base)}
    for tag, w in want.items():
        got = {k: v for k, v in stored[tag].items() if k != "vocab"}
        if got != w:
            print(f"cached '{tag}' was built with different settings; "
                  "loading the source dataset instead")
            print(f"  cached: {got}\n  wanted: {w}")
            return None

    vocab = stored["train"].get("vocab")
    if not isinstance(vocab, dict):
        # Older caches stored only the sorted symbols, which cannot reproduce the
        # token->id mapping the cached labels were written against.
        print("cached manifest predates vocab storage; rebuilding from source")
        return None

    print(f"reusing cached features from {args.cache_dir} (source not needed)")
    tokenizer = tokenizer_from_vocab(vocab, args.output_dir)
    fe = transformers.AutoFeatureExtractor.from_pretrained(MODEL_ID)
    processor = transformers.Wav2Vec2BertProcessor(feature_extractor=fe,
                                                   tokenizer=tokenizer)

    def load(tag):
        return datasets.concatenate_datasets(
            [datasets.Dataset.from_file(str(f))
             for f in shard_files(args.cache_dir, tag)])

    train_ds, valid_ds = load("train"), load("valid")
    print(f"  train={len(train_ds):,}  valid={len(valid_ds):,}")

    # References come back by decoding the cached labels -- the transcriptions
    # live only in the source, which we deliberately did not load.
    valid_refs = [tokenizer.decode(ids, group_tokens=False)
                  for ids in valid_ds["labels"]]
    return train_ds, valid_ds, processor, valid_refs, valid_ds["language"]


class Collator:
    """Pads audio features and labels independently; masks label padding to -100."""

    def __init__(self, processor):
        self.p = processor

    def __call__(self, features):
        # Features are cached as float16 to halve disk and read bandwidth;
        # restore float32 here so the model and padding logic are unaffected.
        audio = self.p.pad(
            [{"input_features": np.asarray(f["input_features"], dtype=np.float32)}
             for f in features],
            padding=True, return_tensors="pt",
        )
        labels = self.p.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features],
            padding=True, return_tensors="pt",
        )
        # CTC loss ignores -100, so padded label positions contribute nothing.
        audio["labels"] = labels["input_ids"].masked_fill(
            labels.attention_mask.ne(1), -100
        )
        return audio


class LengthGroupedSampler(torch.utils.data.Sampler):
    """Batch clips of similar duration together.

    transformers 5.x dropped `group_by_length`, and without it every batch is
    padded to its longest member -- with clips ranging from 0.5s to 30s that is a
    large fraction of the compute spent on padding. Shuffle, cut into megabatches,
    sort each by length, then shuffle the batch order: near-uniform batches while
    keeping enough randomness that the model doesn't see the same grouping twice.
    """

    def __init__(self, lengths: list[int], batch_size: int, seed: int = 42,
                 megabatch_mult: int = 50):
        self.lengths = lengths
        self.batch_size = batch_size
        self.megabatch_size = batch_size * megabatch_mult
        self.seed = seed
        self.epoch = 0

    def __len__(self) -> int:
        return len(self.lengths)

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch      # reshuffles grouping each epoch

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch)
        indices = torch.randperm(len(self.lengths), generator=g).tolist()

        megabatches = [indices[i:i + self.megabatch_size]
                       for i in range(0, len(indices), self.megabatch_size)]
        # Longest-first inside each megabatch surfaces the worst-case batch early,
        # so an OOM shows up in the first minute rather than an hour in.
        megabatches = [sorted(mb, key=lambda i: self.lengths[i], reverse=True)
                       for mb in megabatches]

        batches = [mb[i:i + self.batch_size]
                   for mb in megabatches
                   for i in range(0, len(mb), self.batch_size)]
        order = torch.randperm(len(batches), generator=g).tolist()
        for b in order:
            yield from batches[b]


class LengthGroupedTrainer(transformers.Trainer):
    """Trainer that uses LengthGroupedSampler when the dataset carries lengths."""

    def _get_train_sampler(self, *args, **kwargs):
        ds = self.train_dataset
        if ds is None or "length" not in getattr(ds, "column_names", []):
            return super()._get_train_sampler(*args, **kwargs)
        return LengthGroupedSampler(
            ds["length"], self.args.per_device_train_batch_size, self.args.seed
        )


def make_training_args(**kwargs) -> transformers.TrainingArguments:
    """Build TrainingArguments, dropping keys this transformers version rejects.

    The argument set churns across releases (v5 removed `group_by_length`, v4.46
    renamed `evaluation_strategy` to `eval_strategy`). Kaggle, Colab and RunPod
    will not agree on a version, so filter against the actual signature rather
    than pinning -- and say out loud what got dropped, since a silently ignored
    argument is how you end up wondering why training is slow.
    """
    import inspect

    sig = inspect.signature(transformers.TrainingArguments.__init__)
    accepted = set(sig.parameters)

    # Renames, newest name first: try each until one is accepted.
    aliases = {
        "eval_strategy": ("eval_strategy", "evaluation_strategy"),
        "warmup_ratio": ("warmup_ratio",),
    }
    resolved, dropped = {}, []
    for key, value in kwargs.items():
        for name in aliases.get(key, (key,)):
            if name in accepted:
                resolved[name] = value
                break
        else:
            dropped.append(key)

    if dropped:
        print(f"note: transformers {transformers.__version__} does not accept "
              f"{dropped} -- proceeding without")
        if "group_by_length" in dropped:
            print("      (length grouping off: more padding waste, same accuracy)")
    return transformers.TrainingArguments(**resolved)


def _resample(arr: np.ndarray, factor: float) -> np.ndarray:
    """Speed up (factor>1) or slow down the waveform, shifting pitch with it.

    Resampling without correcting pitch is deliberate -- it is the classic Kaldi
    speed-perturb recipe, and the pitch shift is what makes it act like extra
    speakers rather than just extra tempo. That is what Phase 2 tests.
    """
    if factor == 1.0:
        return arr
    n = int(round(len(arr) / factor))
    # Linear interpolation onto the resampled grid; good enough for augmentation
    # and avoids a scipy/librosa dependency in the hot path.
    idx = np.linspace(0, len(arr) - 1, n)
    return np.interp(idx, np.arange(len(arr)), arr).astype(np.float32)


def shard_files(cache_dir: Path, tag: str) -> list[Path]:
    """The Arrow files datasets.map wrote for this tag, in order.

    With num_proc>1 it emits `<tag>_00000_of_000NN.arrow` per worker; with a
    single process it writes `<tag>.arrow` directly.
    """
    sharded = sorted(cache_dir.glob(f"{tag}_*_of_*.arrow"))
    if sharded:
        return sharded
    single = cache_dir / f"{tag}.arrow"
    return [single] if single.exists() else []


def prepare(ds, processor, num_proc: int, speeds: tuple[float, ...] = (1.0,),
            writer_batch_size: int = 100, cache_file: Path | None = None,
            in_memory: bool = False):
    """Extract features. Uses the cheaper non-batched path unless augmenting.

    writer_batch_size matters more than it looks: each feature array is ~480KB,
    so the datasets default of 1000 buffers ~480MB per worker before flushing.
    With 16 workers that is ~7.7GB of buffering and very bursty writes -- painful
    on network storage. 100 keeps the flushes small and steady.
    """
    # Belt and braces: the env vars above cover libraries that read them at
    # import, this covers torch regardless of import order. Forked workers
    # inherit it. GPU training does not need CPU threads, so leaving it at 1
    # costs nothing later.
    torch.set_num_threads(1)

    if speeds == (1.0,):
        # No augmentation: one row in, one row out. Avoids the per-row dict
        # wrapping that batched=True with batch_size=1 imposes.
        def fn_single(row):
            arr, sr = wdata.audio_array(row["audio"])
            feats = processor(arr, sampling_rate=sr).input_features[0]
            return {
                # float16 halves the cache (~16GB -> ~8GB) and halves the bytes
                # read per training step. Training runs in bf16 regardless, and
                # the collator restores float32 before the model sees it, so
                # nothing downstream changes.
                "input_features": np.asarray(feats, dtype=np.float16),
                "labels": processor.tokenizer(clean(row["transcription"])).input_ids,
                "language": row["language"],
                "length": len(feats),
            }

        return ds.map(fn_single, remove_columns=ds.column_names, num_proc=num_proc,
                      writer_batch_size=writer_batch_size,
                      keep_in_memory=in_memory,
                      cache_file_name=None if in_memory else (
                          str(cache_file) if cache_file else None),
                      desc="extracting features")

    def fn(batch):
        # batched=True with batch_size=1: every field arrives as a 1-element list.
        arr, sr = wdata.audio_array(batch["audio"][0])
        labels = processor.tokenizer(clean(batch["transcription"][0])).input_ids
        feats, lens = [], []
        for f in speeds:
            x = processor(_resample(arr, f), sampling_rate=sr).input_features[0]
            feats.append(x)
            lens.append(len(x))
        return {
            "input_features": feats,
            "labels": [labels] * len(speeds),      # transcript is unchanged
            "language": [batch["language"][0]] * len(speeds),
            "length": lens,                        # drives LengthGroupedSampler
        }

    # batched with a 1-row batch: lets each input emit len(speeds) output rows.
    return ds.map(fn, remove_columns=ds.column_names, num_proc=num_proc,
                  batched=True, batch_size=1,
                  writer_batch_size=writer_batch_size,
                  keep_in_memory=in_memory,
                  cache_file_name=None if in_memory else (
                      str(cache_file) if cache_file else None),
                  desc=f"extracting features (speeds={list(speeds)})")


# ~750 frames x 160 mels x 2 bytes (float16) for a typical 15s clip.
BYTES_PER_ROW = 750 * 160 * 2


def check_disk_space(rows: int, cache_dir: Path) -> None:
    """Fail now rather than at writer.finalize() ten minutes in.

    datasets writes the map output under HF_DATASETS_CACHE and save_to_disk then
    copies it into cache_dir, so the peak is roughly twice the final size.
    """
    import shutil

    need = rows * BYTES_PER_ROW * 2
    cache_dir.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(cache_dir).free
    print(f"disk: {free/1e9:.1f}GB free, ~{need/1e9:.1f}GB needed "
          f"for {rows:,} rows (map output + cache copy)")
    if free < need:
        raise SystemExit(
            f"\nNot enough space: {free/1e9:.1f}GB free, ~{need/1e9:.1f}GB needed.\n"
            f"  - free the downloaded parquet with --free-download-cache (~13GB)\n"
            f"  - or point --cache-dir at a larger disk\n"
            f"  - or provision a bigger volume\n"
            f"Extraction would otherwise fail at the final flush with "
            f"'OSError: [Errno 5] Input/output error'."
        )


def prepare_cached(ds, processor, num_proc: int, cache_dir: Path | None, tag: str,
                   key: dict, speeds: tuple[float, ...] = (1.0,),
                   in_memory: bool = False):
    """Feature extraction with an explicit on-disk cache.

    datasets.map caches by fingerprint, but the fingerprint hashes the mapped
    function *and its closure* -- which here includes the processor. Any edit to
    this file, or an unstable hash of the processor, silently invalidates it and
    you pay full extraction again. At ~40 minutes a run that is too expensive to
    leave to chance, so we save explicitly and validate against the parameters
    that actually affect the output.
    """
    if in_memory:
        # No filesystem at all. ~8.3GB of features against 117GB of RAM, and it
        # sidesteps storage that fails on close (RunPod's MooseFS: Errno 5 at
        # writer.finalize(), whichever component happens to write last). Costs a
        # re-extraction each run -- ~10 min -- which beats a run that dies.
        print("extracting to memory (no cache; re-extracts each run)")
        return prepare(ds, processor, num_proc, speeds, in_memory=True)

    if cache_dir is None:
        return prepare(ds, processor, num_proc, speeds)

    check_disk_space(len(ds) * len(speeds), cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest = cache_dir / f"{tag}.json"

    existing = shard_files(cache_dir, tag)
    if existing and manifest.exists():
        stored = json.loads(manifest.read_text())
        if stored == key:
            print(f"reusing cached features: {len(existing)} shard(s) in {cache_dir}")
            return datasets.concatenate_datasets(
                [datasets.Dataset.from_file(str(f)) for f in existing])
        # Stale cache is worse than none -- it trains on the wrong thing silently.
        print(f"cache for '{tag}' was built with different settings, rebuilding")
        print(f"  cached: {stored}\n  wanted: {key}")
        for f in existing:
            f.unlink()

    # Write the map output straight into cache_dir instead of letting datasets
    # put it under HF_DATASETS_CACHE and then copying it with save_to_disk. That
    # copy doubled peak disk *and* was the step that failed with Errno 5 on
    # RunPod's network volume -- the map itself always succeeded. Reading the
    # shards back with from_file gives the identical dataset.
    out = prepare(ds, processor, num_proc, speeds,
                  cache_file=cache_dir / f"{tag}.arrow")
    manifest.write_text(json.dumps(key, indent=1, sort_keys=True))
    written = shard_files(cache_dir, tag)
    total = sum(f.stat().st_size for f in written)
    print(f"cached features -> {cache_dir} "
          f"({len(written)} shard(s), {total/1e9:.1f}GB)")
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--output-dir", type=Path, required=True)
    ap.add_argument("--epochs", type=float, default=6.0)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--grad-accum", type=int, default=4)
    ap.add_argument("--lr", type=float, default=5e-5)
    ap.add_argument("--num-proc", type=int, default=8,
                    help="workers for feature extraction (CPU-bound; use all cores)")
    ap.add_argument("--gradient-checkpointing", default=None,
                    action=argparse.BooleanOptionalAction,
                    help="default: off when the GPU has >40GB, on otherwise")
    ap.add_argument("--in-memory", action="store_true",
                    help="extract features into RAM, never touching a filesystem. "
                         "~8.3GB for this dataset. Use when storage is unreliable "
                         "(RunPod network volumes fail on close with Errno 5). "
                         "Costs a ~10min re-extraction every run")
    ap.add_argument("--free-download-cache", action="store_true",
                    help="delete the downloaded parquet (~13GB) once it has been "
                         "converted to Arrow, before extraction writes its output. "
                         "Re-downloads if you need it again")
    ap.add_argument("--load-proc", type=int, default=1,
                    help="workers for dataset download/Arrow generation. Keep low: "
                         "it is I/O-bound, and high values make datasets' forked "
                         "workers fail with 'I/O operation on closed file'")
    ap.add_argument("--valid-frac", type=float, default=0.06)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--limit", type=int, default=0, help="debug: cap rows loaded")
    ap.add_argument("--max-s", type=float, default=30.0,
                    help="drop clips longer than this. Attention is quadratic in "
                         "length, so a few long clips cost disproportionately")
    ap.add_argument("--min-s", type=float, default=0.5)
    ap.add_argument("--max-frames", type=int, default=0,
                    help="drop cached training clips longer than this many frames "
                         "(~50/sec, so 1000 ~= 20s). Applied after loading, so it "
                         "needs no re-extraction and does not change the cache key. "
                         "The main lever against OOM: attention is quadratic here")
    ap.add_argument("--cache-dir", type=Path, default=None,
                    help="persist extracted features here so a failed run doesn't "
                         "repeat the ~40min extraction; put it on a persistent volume")
    ap.add_argument("--save-strategy", choices=("steps", "epoch"), default="epoch",
                    help="epoch: one checkpoint per epoch (default)")
    ap.add_argument("--save-steps", type=int, default=500,
                    help="only used with --save-strategy steps")
    ap.add_argument("--push-to-hub", type=str, nargs="?", const="auto", default="",
                    help=f"upload checkpoints as they are saved, surviving pod "
                         f"loss. Bare flag uses {HUB_USER}/<output-dir-name>; pass "
                         f"a value to override, e.g. {HUB_USER}/waxal-ctc-v1")
    ap.add_argument("--hub-strategy", choices=("every_save", "end"),
                    default="every_save",
                    help="every_save: upload each epoch (~5GB, insurance against "
                         "losing the pod, but can stall training on a slow uplink). "
                         "end: upload once when training finishes")
    ap.add_argument("--hub-public", action="store_true",
                    help="publish the Hub repo publicly. Off by default: the rules "
                         "forbid sharing work outside your team during the challenge")
    ap.add_argument("--resume", action="store_true",
                    help="resume from the last checkpoint in --output-dir, "
                         "restoring optimizer and LR schedule")
    ap.add_argument("--init-from", type=str, default="",
                    help="continue from a finished run: a local checkpoint or Hub "
                         "repo id (e.g. ngia/ctc-v1). Loads the weights but starts "
                         "a fresh optimizer and LR schedule, which --resume does "
                         "not -- a completed schedule has decayed to zero. Lower "
                         "--lr (2e-5) suits an already-trained model")
    # SpecAugment. Applied inside the model at training time only, so it costs
    # nothing extra and does NOT invalidate the feature cache -- tune these freely.
    ap.add_argument("--mask-time-prob", type=float, default=0.05,
                    help="fraction of time steps masked (0.05-0.10 typical)")
    ap.add_argument("--mask-time-length", type=int, default=10)
    ap.add_argument("--mask-feature-prob", type=float, default=0.0,
                    help="frequency masking; 0.01-0.05 helps on unseen speakers")
    ap.add_argument("--mask-feature-length", type=int, default=64)
    # Speed perturbation. Changes the audio, so it DOES invalidate the cache and
    # multiplies extraction time and disk by len(factors).
    ap.add_argument("--speed-perturb", type=str, default="",
                    help="comma-separated factors, e.g. 0.9,1.0,1.1 -- triples the "
                         "training set with faster/slower copies")
    args = ap.parse_args()

    transformers.set_seed(args.seed)          # rules require reproducibility

    if args.gradient_checkpointing is None:
        args.gradient_checkpointing = hw.wants_gradient_checkpointing()
    print(f"{hw.describe()}  vram={hw.vram_gb():.0f}GB  "
          f"gradient_checkpointing={args.gradient_checkpointing}")

    if args.push_to_hub == "auto":
        args.push_to_hub = f"{HUB_USER}/{args.output_dir.name}"
    if args.push_to_hub:
        vis = "public" if args.hub_public else "private"
        print(f"checkpoints -> huggingface.co/{args.push_to_hub} ({vis})")

    # Everything that decides which rows exist and what the features look like.
    # Learning rate and epochs are deliberately absent: they don't affect
    # features, so re-tuning them reuses the cache.
    key_base = {
        "model": MODEL_ID, "langs": list(wdata.LANGS), "sr": wdata.SR,
        "valid_frac": args.valid_frac, "seed": args.seed, "limit": args.limit,
        "min_s": args.min_s, "max_s": args.max_s,
    }

    cached = load_from_cache_only(args, key_base)
    if cached is not None:
        train_ds, valid_ds, processor, valid_refs, valid_langs = cached
    else:
        train_ds, valid_ds, processor, valid_refs, valid_langs = \
            build_from_source(args, key_base)

    if args.max_frames:
        # Attention memory grows with the square of sequence length, so a small
        # tail of long clips sets the peak for the whole run. Filtering here
        # rather than at extraction means no re-extraction and no cache-key
        # change -- the cache keeps every row, this is just a view of it.
        before = len(train_ds)
        train_ds = train_ds.filter(lambda n: n <= args.max_frames,
                                   input_columns="length")
        dropped = before - len(train_ds)
        print(f"max-frames {args.max_frames}: train {before:,} -> {len(train_ds):,} "
              f"({dropped:,} dropped, {100*dropped/before:.1f}%)")
        # Validation is left intact: eval runs under no_grad, so it holds no
        # activations and is not what drives the peak.

    # Save the processor up front, not at the end. Trainer's checkpoints contain
    # only weights, so a run that is interrupted -- or whose final save does not
    # complete -- leaves checkpoints that cannot be loaded for inference. Written
    # here, every checkpoint's parent directory has what it needs.
    processor.save_pretrained(str(args.output_dir))
    print(f"processor -> {args.output_dir}")

    tokenizer = processor.tokenizer
    build_and_train(args, processor, tokenizer, train_ds, valid_ds,
                    valid_refs, valid_langs)


def build_from_source(args, key_base: dict):
    """Download, filter, split, and extract features from the source dataset."""
    print("loading labeled data (train+validation only; test is off-limits)")
    # With --limit, fetch one parquet shard per language instead of all ~12.6GB.
    ds = wdata.load_labeled(num_proc=args.load_proc, shards=1 if args.limit else 0)
    if args.limit:
        # Shuffle first: the shards concatenate language by language, so taking
        # the head would give an all-Lingala "smoke test" that never exercises
        # the other two languages or the joint vocabulary.
        ds = ds.shuffle(seed=args.seed).select(range(min(args.limit, len(ds))))
    print(f"  {len(ds):,} rows")

    ds = wdata.filter_usable(ds, min_s=args.min_s, max_s=args.max_s)
    print(f"  {len(ds):,} usable (clips {args.min_s}-{args.max_s}s)")

    if args.free_download_cache:
        # load_dataset has already materialized everything into Arrow, so the
        # downloaded parquet is dead weight -- and it is ~13GB competing for the
        # same disk that extraction is about to write to.
        import shutil
        hub = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface")) / "hub"
        repo = hub / f"datasets--{wdata.HF_REPO.replace('/', '--')}"
        if repo.exists():
            freed = sum(f.stat().st_size for f in repo.rglob("*") if f.is_file())
            shutil.rmtree(repo, ignore_errors=True)
            print(f"freed {freed/1e9:.1f}GB of downloaded parquet ({repo})")
        else:
            print(f"nothing to free at {repo}")

    print("building speaker-disjoint validation split")
    split = wdata.speaker_disjoint_split(ds, args.valid_frac, args.seed)
    print(f"  {split}")

    vocab = build_vocab(split.train["transcription"])
    tokenizer = tokenizer_from_vocab(vocab, args.output_dir)
    fe = transformers.AutoFeatureExtractor.from_pretrained(MODEL_ID)
    processor = transformers.Wav2Vec2BertProcessor(feature_extractor=fe, tokenizer=tokenizer)

    speeds = tuple(float(s) for s in args.speed_perturb.split(",")) \
        if args.speed_perturb else (1.0,)

    # The full token->id map, not just the symbols: the cached labels were
    # written against these exact ids, so a reload must reproduce them.
    key = {**key_base, "vocab": vocab}
    # Augment training only. A perturbed validation set would measure the model
    # on audio the Phase 2 set will never contain, and would not be comparable
    # to the un-augmented baseline.
    train_ds = prepare_cached(split.train, processor, args.num_proc,
                              args.cache_dir, "train",
                              {**key, "speeds": list(speeds)}, speeds,
                              in_memory=args.in_memory)
    valid_ds = prepare_cached(split.valid, processor, args.num_proc,
                              args.cache_dir, "valid", key,
                              in_memory=args.in_memory)
    if speeds != (1.0,):
        print(f"speed perturbation {list(speeds)}: train {len(split.train):,} "
              f"-> {len(train_ds):,} rows")
    valid_langs = valid_ds["language"]
    valid_refs = [clean(t) for t in split.valid["transcription"]]
    return train_ds, valid_ds, processor, valid_refs, valid_langs


def build_and_train(args, processor, tokenizer, train_ds, valid_ds,
                    valid_refs, valid_langs) -> None:
    # --init-from continues an earlier run: load its weights but start a fresh
    # optimizer and LR schedule. Distinct from --resume, which restores the old
    # schedule too -- and a finished run's schedule has already decayed to zero,
    # so resuming it would train at effectively no learning rate.
    init_from = args.init_from or MODEL_ID
    if args.init_from:
        print(f"warm start from {args.init_from} (fresh optimizer and schedule)")
    model = transformers.Wav2Vec2BertForCTC.from_pretrained(
        init_from,
        attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0,
        # SpecAugment-style masking is the main regularizer here; the labeled
        # set is small relative to the 580M encoder.
        mask_time_prob=args.mask_time_prob,
        mask_time_length=args.mask_time_length,
        mask_feature_prob=args.mask_feature_prob,
        mask_feature_length=args.mask_feature_length,
        layerdrop=0.0,
        ctc_loss_reduction="mean",
        add_adapter=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
    )

    def compute_metrics(pred):
        ids = np.argmax(pred.predictions, axis=-1)
        hyps = processor.batch_decode(ids)
        s = score(valid_refs, hyps)
        per = score_by_language(valid_refs, hyps, valid_langs)
        # Zindi does not publish whether it averages per utterance or computes
        # corpus-level rates. They diverge sharply here -- short clips carry
        # huge per-utterance WER -- so log both and compare against the
        # leaderboard to learn which one we are actually being scored on.
        out = {
            "wer": s.wer, "cer": s.cer, "combined": s.combined,
            "combined_mean": s.combined_mean,
            "wer_mean": s.wer_mean, "cer_mean": s.cer_mean,
        }
        out.update({f"combined_{l}": v.combined for l, v in per.items()})
        return out

    targs = make_training_args(
        output_dir=str(args.output_dir),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        # NOT torch.cuda.is_bf16_supported(): that reports True on a T4, where
        # bf16 is emulated and much slower than the card's fp16 tensor cores.
        bf16=hw.supports_bf16(),
        fp16=not hw.supports_bf16(),
        # Checkpointing trades ~35% throughput for memory. It is mandatory on a
        # 16GB T4 (measured: OOM without it at batch 4) and pure waste on an 80GB
        # A100, so decide from the card rather than hardcoding the small-GPU case.
        gradient_checkpointing=args.gradient_checkpointing,
        # Evaluate and save on the same cadence so load_best_model_at_end always
        # has a metric for every checkpoint it might pick.
        eval_strategy=args.save_strategy,
        save_strategy=args.save_strategy,
        **({"eval_steps": args.save_steps, "save_steps": args.save_steps}
           if args.save_strategy == "steps" else {}),
        logging_steps=50,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="combined",
        greater_is_better=False,          # lower WER/CER wins
        # We group by length ourselves (LengthGroupedTrainer); keep the extra
        # columns it needs, since the collator already selects what the model sees.
        remove_unused_columns=False,
        dataloader_num_workers=4,
        seed=args.seed,
        report_to=[],
        # "every_save" uploads each checkpoint as it is written, so a pod that
        # dies mid-run costs one epoch rather than the whole run.
        **({"push_to_hub": True,
            "hub_model_id": args.push_to_hub,
            "hub_strategy": args.hub_strategy,
            "hub_private_repo": not args.hub_public} if args.push_to_hub else {}),
    )

    trainer = LengthGroupedTrainer(
        model=model, args=targs,
        train_dataset=train_ds, eval_dataset=valid_ds,
        data_collator=Collator(processor),
        compute_metrics=compute_metrics,
    )
    resume = None
    if args.resume:
        last = transformers.trainer_utils.get_last_checkpoint(str(args.output_dir))
        if last:
            print(f"resuming from {last}")
            resume = last
        else:
            print(f"--resume given but no checkpoint in {args.output_dir}; "
                  "starting fresh")

    trainer.train(resume_from_checkpoint=resume)

    best = args.output_dir / "best"
    trainer.save_model(str(best))
    processor.save_pretrained(str(best))
    print(f"saved -> {best}")

    if args.push_to_hub:
        # The per-checkpoint uploads carry model weights but not the processor;
        # without it the repo can't tokenize or extract features on its own.
        processor.push_to_hub(args.push_to_hub, private=not args.hub_public)
        trainer.push_to_hub(commit_message="final best checkpoint")
        print(f"pushed -> https://huggingface.co/{args.push_to_hub}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/infer.py
"""Run a trained CTC model over evaluation audio and write a Zindi submission.

Phase 1 mode reads the public test audio but never its labels (see
waxal.data.load_test_audio). Phase 2 mode points at whatever directory the
organizers publish.

    python scripts/infer.py --model out/ctc-v1/best --phase 1 --out sub.csv
"""

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import pandas as pd
import torch
import transformers

from waxal import data as wdata


MODEL_ID = "facebook/w2v-bert-2.0"


def load_processor(model_dir: Path, vocab: Path | None):
    """Load the processor, rebuilding it if the checkpoint lacks one.

    Trainer's intermediate checkpoints hold only model weights -- the processor
    is written by save_pretrained at the end of training. Since the feature
    extractor is just the pretrained one and the tokenizer is fully determined
    by vocab.json (which training writes to --output-dir), we can reconstruct it
    from the checkpoint's parent rather than requiring a finished run.
    """
    try:
        return transformers.Wav2Vec2BertProcessor.from_pretrained(str(model_dir))
    except OSError:
        pass

    candidates = [vocab] if vocab else []
    candidates += [model_dir / "vocab.json", model_dir.parent / "vocab.json"]
    found = next((p for p in candidates if p and p.exists()), None)

    if found is None and not model_dir.exists() and "/" in str(model_dir):
        # Looks like a Hub repo id rather than a local path. A repo whose
        # processor push never completed can still carry vocab.json, which is
        # all the tokenizer needs.
        try:
            from huggingface_hub import hf_hub_download
            found = Path(hf_hub_download(str(model_dir), "vocab.json"))
            print(f"fetched vocab.json from the Hub repo {model_dir}")
        except Exception as e:
            print(f"could not fetch vocab.json from {model_dir}: "
                  f"{type(e).__name__}: {e}")

    if found is None:
        raise SystemExit(
            f"no processor for {model_dir}, and no vocab.json locally or on the Hub.\n"
            f"Pass --vocab /path/to/vocab.json (training writes it to --output-dir)."
        )

    print(f"checkpoint has no processor; rebuilding from {found}")
    tokenizer = transformers.Wav2Vec2CTCTokenizer(
        str(found), unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")
    fe = transformers.AutoFeatureExtractor.from_pretrained(MODEL_ID)
    return transformers.Wav2Vec2BertProcessor(feature_extractor=fe,
                                              tokenizer=tokenizer)


def _ctc_labels(processor) -> list[str]:
    """The alphabet as pyctcdecode wants it: one entry per token id.

    pyctcdecode requires single-character labels -- it treats any multi-character
    entry as BPE-style tokenization and decodes accordingly. Our vocabulary has
    three multi-character tokens ([UNK], plus <s>/</s> which the tokenizer appends
    automatically), so they are mapped to single placeholder characters that
    cannot occur in the transcripts. They are never predicted anyway: the model
    has no training signal for them.
    """
    vocab = processor.tokenizer.get_vocab()
    labels = [tok for tok, _ in sorted(vocab.items(), key=lambda kv: kv[1])]

    placeholders = iter("\u2047\u2048\u2049\u204a\u204b")
    out = []
    for tok in labels:
        if tok == "[PAD]":
            out.append("")          # CTC blank
        elif tok == "|":
            out.append(" ")         # word delimiter
        elif len(tok) > 1:
            out.append(next(placeholders))
        else:
            out.append(tok)
    return out


def build_lm_decoder(processor, lm: Path, unigrams: Path | None,
                     alpha: float, beta: float):
    """Beam decoder over the model's own alphabet, scored by an n-gram LM.

    pyctcdecode expects the alphabet ordered by token id, with the CTC blank as
    "" and the word delimiter as a literal space.
    """
    from pyctcdecode import build_ctcdecoder

    labels = _ctc_labels(processor)

    words = None
    if unigrams and unigrams.exists():
        words = [w for w in unigrams.read_text().split("\n") if w]

    return build_ctcdecoder(labels, kenlm_model_path=str(lm), unigrams=words,
                            alpha=alpha, beta=beta)


@torch.no_grad()
def transcribe(ds, model, processor, device, batch_size: int = 8,
               decoder=None) -> dict[str, str]:
    model.eval().to(device)
    out: dict[str, str] = {}
    for start in range(0, len(ds), batch_size):
        rows = ds[start:start + batch_size]
        arrays = [wdata.audio_array(a)[0] for a in rows["audio"]]
        feats = processor(
            arrays, sampling_rate=wdata.SR, return_tensors="pt", padding=True,
        ).to(device)
        with torch.autocast(device_type=device.type,
                            dtype=torch.bfloat16, enabled=device.type == "cuda"):
            logits = model(**feats).logits
        if decoder is None:
            hyps = processor.batch_decode(logits.argmax(-1).cpu().numpy())
        else:
            # Beam search needs the full distribution, and pyctcdecode is
            # float32-only. This runs on CPU, so it is the slow part.
            lp = logits.float().cpu().numpy()
            hyps = [decoder.decode(lp[i]) for i in range(lp.shape[0])]
        out.update(zip(rows["id"], hyps))
        if start % (batch_size * 25) == 0:
            print(f"  {min(start + batch_size, len(ds)):,}/{len(ds):,}", flush=True)
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", type=Path, required=True)
    ap.add_argument("--phase", type=int, choices=(1, 2), default=1)
    ap.add_argument("--phase2-dir", type=str, default="")
    ap.add_argument("--sample-submission", type=Path,
                    default=Path("data/raw/SampleSubmission.csv"))
    ap.add_argument("--vocab", type=Path, default=None,
                    help="vocab.json to rebuild the tokenizer from, if the "
                         "checkpoint has no processor. Defaults to looking in the "
                         "model dir and its parent")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--num-proc", type=int, default=1,
                    help="dataset loading workers; keep low (see train_ctc.py)")
    ap.add_argument("--lm", type=Path, default=None,
                    help="KenLM .arpa from build_lm.py; switches greedy decoding "
                         "for beam search. Needs pyctcdecode and kenlm installed")
    ap.add_argument("--unigrams", type=Path, default=None)
    ap.add_argument("--alpha", type=float, default=0.5, help="LM weight")
    ap.add_argument("--beta", type=float, default=1.5,
                    help="word insertion bonus")
    args = ap.parse_args()

    processor = load_processor(args.model, args.vocab)
    model = transformers.Wav2Vec2BertForCTC.from_pretrained(str(args.model))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if args.phase == 1:
        ds = wdata.load_test_audio(num_proc=args.num_proc)
    else:
        if not args.phase2_dir:
            ap.error("--phase 2 requires --phase2-dir")
        ds = wdata.load_phase2_audio(args.phase2_dir, num_proc=args.num_proc)
    print(f"transcribing {len(ds):,} clips on {device}")

    decoder = None
    if args.lm:
        decoder = build_lm_decoder(processor, args.lm, args.unigrams,
                                   args.alpha, args.beta)
        print(f"beam decoding with {args.lm} "
              f"(alpha={args.alpha}, beta={args.beta})")

    preds = transcribe(ds, model, processor, device, args.batch_size, decoder)

    if args.sample_submission and args.sample_submission.exists():
        sub = pd.read_csv(args.sample_submission, escapechar="\\")
        sub["Target"] = sub.ID.map(preds)
    else:
        # SampleSubmission.csv is gitignored, so it is often absent on a rented
        # box. The ids we just transcribed are the same set, and Zindi matches
        # on the ID column rather than row order.
        print(f"no sample submission at {args.sample_submission} -- "
              "building from the transcribed ids")
        sub = pd.DataFrame({"ID": list(preds), "Target": list(preds.values())})

    missing = sub.Target.isna().sum()
    if missing:
        # A blank target still scores (badly) -- a missing row may not score at all.
        print(f"WARNING: {missing} ids had no prediction; filling with empty string")
        sub["Target"] = sub.Target.fillna("")

    empty = (sub.Target.str.strip() == "").sum()
    print(f"empty predictions: {empty:,}/{len(sub):,}")

    args.out.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(args.out, index=False)
    print(f"wrote {args.out}  ({len(sub):,} rows)")
    print(sub.head(3).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/bench.py
"""Find out where training time actually goes before renting a bigger GPU.

Observed 106 s/it on a T4 is far off what the arithmetic predicts, so the
bottleneck may not be compute at all. This times the two halves separately:

  * GPU: forward+backward on synthetic tensors, no data pipeline involved.
  * Data: iterating the real dataloader, no model involved.

If GPU time dominates, a faster card fixes it. If data time dominates, a faster
card changes nothing and the fix is in the pipeline.

    python scripts/bench.py --cache-dir /workspace/cache --batch-size 4
"""

import argparse
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import datasets
import torch
import transformers

from waxal import hw

MODEL_ID = "facebook/w2v-bert-2.0"


def bench_gpu(batch_size: int, frames: int, vocab: int, steps: int,
              checkpointing: bool, dtype: torch.dtype) -> float:
    """Seconds per optimizer step, synthetic data, nothing else in the way."""
    dev = torch.device("cuda")
    model = transformers.Wav2Vec2BertForCTC.from_pretrained(
        MODEL_ID, vocab_size=vocab, ctc_loss_reduction="mean",
        add_adapter=True, pad_token_id=vocab - 1,
    ).to(dev)
    if checkpointing:
        model.gradient_checkpointing_enable()
    model.train()

    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
    amp = dtype
    scaler = torch.amp.GradScaler(enabled=amp == torch.float16)

    x = torch.randn(batch_size, frames, 160, device=dev)
    y = torch.randint(0, vocab - 2, (batch_size, frames // 12), device=dev)

    for i in range(steps + 2):          # first two warm up cudnn/allocator
        if i == 2:
            torch.cuda.synchronize()
            t0 = time.perf_counter()
        with torch.autocast("cuda", dtype=amp):
            loss = model(input_features=x, labels=y).loss
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / steps


def bench_data(cache_dir: Path, batch_size: int, workers: int, steps: int) -> tuple:
    """Seconds per batch pulled through the real dataloader, model excluded."""
    # The cache is datasets.map output written straight into cache_dir as
    # `train_00000_of_000NN.arrow` shards -- not a save_to_disk directory.
    shards = sorted(cache_dir.glob("train_*_of_*.arrow")) or \
        [p for p in [cache_dir / "train.arrow"] if p.exists()]
    if not shards:
        raise FileNotFoundError(f"no feature shards in {cache_dir}")
    ds = datasets.concatenate_datasets(
        [datasets.Dataset.from_file(str(f)) for f in shards])
    lengths = ds["length"]

    def collate(feats):
        import numpy as np
        m = max(len(f["input_features"]) for f in feats)
        out = np.zeros((len(feats), m, 160), dtype=np.float32)
        for i, f in enumerate(feats):
            a = np.asarray(f["input_features"], dtype=np.float32)
            out[i, :len(a)] = a
        return torch.from_numpy(out)

    dl = torch.utils.data.DataLoader(
        ds, batch_size=batch_size, collate_fn=collate,
        num_workers=workers, shuffle=True,
    )
    it = iter(dl)
    next(it)                              # warm up workers
    t0 = time.perf_counter()
    n = 0
    for _ in range(steps):
        try:
            next(it)
        except StopIteration:
            break
        n += 1
    return (time.perf_counter() - t0) / max(n, 1), lengths


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache-dir", type=Path, default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    ap.add_argument("--grad-accum", type=int, default=8)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--steps", type=int, default=5)
    ap.add_argument("--vocab", type=int, default=85)
    args = ap.parse_args()

    if not torch.cuda.is_available():
        print("no GPU visible")
        return
    print(f"GPU: {hw.describe()}\n")

    have_cache = bool(args.cache_dir) and bool(
        list(args.cache_dir.glob("train_*_of_*.arrow"))
        or (args.cache_dir / "train.arrow").exists())
    if args.cache_dir and not have_cache:
        print(f"no feature shards in {args.cache_dir} -- "
              "running the GPU benchmark only\n")
    if have_cache:
        per_batch, lengths = bench_data(args.cache_dir, args.batch_size,
                                        args.workers, 20)
        import statistics
        print("clip length in frames (100 frames ~= 1s of audio):")
        print(f"  mean {statistics.mean(lengths):.0f}  median "
              f"{statistics.median(lengths):.0f}  "
              f"p95 {sorted(lengths)[int(len(lengths)*0.95)]}  max {max(lengths)}")
        data_step = per_batch * args.grad_accum
        print(f"\ndata only: {per_batch:.3f}s/batch -> {data_step:.1f}s per "
              f"optimizer step ({args.grad_accum} accum)\n")
        frames = int(statistics.mean(lengths))
    else:
        data_step, frames = 0.0, 750
        print("(no --cache-dir: skipping data benchmark)\n")

    total = 0.0
    for dtype, label in ((torch.float16, "fp16"), (torch.bfloat16, "bf16")):
        for ckpt in (True, False):
            try:
                s = bench_gpu(args.batch_size, frames, args.vocab, args.steps,
                              ckpt, dtype)
                total = s * args.grad_accum
                print(f"gpu {label} checkpointing={str(ckpt):5s}: "
                      f"{s:.3f}s/microbatch -> {total:.1f}s per optimizer step")
            except torch.OutOfMemoryError:
                # Expected without checkpointing on a small card; not a failure.
                print(f"gpu {label} checkpointing={str(ckpt):5s}: OOM at "
                      f"batch {args.batch_size} x {frames} frames")
            finally:
                torch.cuda.empty_cache()

    if data_step:
        print(f"\nverdict: data {data_step:.1f}s vs gpu {total:.1f}s per step -> "
              f"{'DATA-BOUND (a faster GPU will not help)' if data_step > total else 'GPU-BOUND (a faster card helps)'}")


if __name__ == "__main__":
    main()

In [ ]:
import sys
sys.path.insert(0, "src")
from waxal.normalize import clean
from waxal.metric import score
assert clean("  Ndaba «x» 12 ⭐️  ") == 'Ndaba "x"'
print("modules OK")

## 2b. Inference only — score a trained model from the Hub

Run **just this section** (cells 1–2 then here) to produce a submission from an
already-trained model. No training, no feature extraction: it pulls the weights
from the Hub and transcribes the Phase 1 test audio (~1.3 GB download).

Needs `HF_TOKEN` in Add-ons → Secrets if the model repo is private.

In [ ]:
# The only thing to change: which model to score with.
#   ngia/ctc-v1     epoch 5, validation 0.1617   <- best so far
#   ngia/ctc-v2     full-data run, epoch 2, 0.1747
#   ngia/ctc-v3     v1 + speed perturbation
MODEL = "ngia/ctc-v2"
OUT = "/kaggle/working/submission.csv"

In [ ]:
!python scripts/infer.py \
    --model {MODEL} \
    --phase 1 \
    --sample-submission /kaggle/input/waxal-csvs/SampleSubmission.csv \
    --out {OUT} \
    --batch-size 16

In [ ]:
import pandas as pd
sub = pd.read_csv(OUT, escapechar="\\")
print(f"{MODEL}: {sub.shape} {list(sub.columns)}")
assert list(sub.columns) == ["ID", "Target"], sub.columns
empty = (sub.Target.fillna("").str.strip() == "").sum()
print(f"empty targets: {empty}/{len(sub)}")
# Degenerate CTC output ("Muta a a a a") shows up as repeated single letters.
degen = sub.Target.fillna("").str.contains(r"(?:\b(\w)\b[ .]*){4,}", regex=True)
print(f"degenerate repeats: {degen.sum()} ({100*degen.mean():.1f}%)")
sub.head(10)

## 2c. Continue training an earlier run

Warm-starts from a finished model on the Hub: loads its weights, then trains
with a **fresh optimizer and LR schedule**. That is what you want after a run
completes — `--resume` would restore the old schedule, which has already decayed
to zero.

Pull the cached features first so this skips the 12.6 GB download and the ~10 min
feature extraction entirely. Use a **lower learning rate** than the original run:
the model is already trained, and 5e-5 would undo some of that.

Kaggle GPU sessions cap at ~9 hours, so size `--epochs` to fit.

In [ ]:
!python scripts/sync_features.py pull --cache-dir /kaggle/temp/cache

!python scripts/train_ctc.py \
    --init-from ngia/ctc-v1 \
    --output-dir /kaggle/working/ctc-v1b \
    --cache-dir /kaggle/temp/cache \
    --push-to-hub ngia/ctc-v1b \
    --hub-strategy end \
    --epochs 3 --batch-size 4 --grad-accum 8 --lr 2e-5 --seed 43

## 2d. Speed perturbation — **A100 only, will not fit a Kaggle session**

Each clip is resampled to 0.9x / 1.0x / 1.1x, tripling the training set. Pitch
shifts with speed, which is the point: each copy behaves like a different
speaker, and Phase 2 is scored on unheard voices.

Measured costs per epoch, batch 4 / accum 8:

| | rows | Kaggle T4 | A100 |
|---|---|---|---|
| normal | 31,316 | 5.4 h | 1.4 h |
| perturbed | 93,948 | **16.3 h** | 4.2 h |

A Kaggle GPU session caps around 9 hours, so one epoch overruns. It also changes
the cache key, so the pulled cache no longer matches and you would first pay a
12.6 GB download plus ~1.7 h of 3x extraction on a T4.

Run it on the A100 instead:

```bash
python scripts/train_ctc.py \
    --init-from ngia/ctc-v2 \
    --output-dir /dev/shm/ctc-v3 \
    --cache-dir /dev/shm/cache-sp \
    --speed-perturb 0.9,1.0,1.1 \
    --push-to-hub ngia/ctc-v3 --hub-strategy end \
    --epochs 2 --batch-size 4 --grad-accum 8 --lr 2e-5 --seed 44
```

Note the separate `--cache-dir`: perturbed features are a different (3x larger,
~30 GB) cache, and keeping them apart means the unperturbed one stays valid.

## 2e. Pseudo-labelling the unlabeled pool

The labeled set is ~180 hours; the unlabeled pool is ~78 GB across the three
languages. Transcribing it with our best model and training on the confident
outputs is the one remaining technique with the right order of magnitude —
augmentation and checkpoint averaging give 5–10% relative, self-training can
give 20–30%.

Output is a small CSV (id, language, transcription, confidence), **not**
features: re-extracting features on the training GPU is far cheaper than moving
~10 GB of arrays off Kaggle.

Nothing here goes near the test split — the unlabeled pool has no transcriptions
at all, so there is no label to leak.

Each shard is ~0.5 GB per language. Start with 2 shards to see the confidence
distribution before committing a whole session, then raise `--shards` and run
again. On a T4 expect roughly 25–35 minutes per 4,000 clips.

In [ ]:
!python scripts/pseudo_label.py \
    --model ngia/ctc-v2-avg \
    --shards 0 1 \
    --out /kaggle/working/pseudo_00.csv \
    --batch-size 16

In [ ]:
import pandas as pd
p = pd.read_csv("/kaggle/working/pseudo_00.csv")
print(p.shape)
print(p.groupby("language").confidence.describe()[["count","25%","50%","75%"]])
# Pick a threshold from this: high confidence means the model was decisive at
# every frame, which correlates with the transcription being right.
for q in (0.5, 0.6, 0.7, 0.8):
    print(f"  conf >= {q}: {(p.confidence >= q).sum():,} rows "
          f"({100*(p.confidence >= q).mean():.0f}%)")
p.sort_values("confidence", ascending=False).head(3)[["language","confidence","transcription"]]

In [ ]:
# Push to the Hub so the training pod can pull it.
from huggingface_hub import HfApi
api = HfApi()
api.create_repo("ngia/waxal-pseudo", repo_type="dataset", private=True, exist_ok=True)
api.upload_file(path_or_fileobj="/kaggle/working/pseudo_00.csv",
                path_in_repo="pseudo_00.csv",
                repo_id="ngia/waxal-pseudo", repo_type="dataset")
print("pushed -> https://huggingface.co/datasets/ngia/waxal-pseudo")

## 3. Smoke test

A few hundred rows end-to-end first. The full run costs hours; a typo shouldn't
cost you one of them. Expect a *terrible* score here — 200 rows trains nothing.
What matters is that it completes without raising.

In [ ]:
!python scripts/train_ctc.py \
    --output-dir /kaggle/temp/smoke \
    --limit 200 --epochs 1 --batch-size 2 --grad-accum 1 \
    --valid-frac 0.25 --num-proc 2

## 4. Full training run

Kaggle sessions are capped at 12 hours (9 for GPU) and the weekly GPU quota is 30
hours, so this is sized to fit one session rather than to be optimal — treat it as
a baseline to beat on RunPod, not the final model.

Sized for a **T4 (15 GB, no bf16)**: batch 4 with 8 accumulation steps holds the
effective batch at 32 while keeping activations inside memory. A 580M model in
mixed precision spends ~10 GB on weights, the fp32 master copy, and AdamW moments
before a single activation is stored.

`CUDA_VISIBLE_DEVICES=0` pins this to one GPU on purpose. With both visible, the
Trainer silently switches to DataParallel, which changes the effective batch size
and adds a failure mode to debug on the first real run. Drop the prefix to use
both once a single-GPU run is known good.

`group_by_length` matters here: these clips vary a lot in duration, and batching
similar lengths together cuts padding waste substantially.

In [ ]:
# --cache-dir keeps the ~40min feature extraction across retries. /kaggle/temp
# dies with the session but /kaggle/working is capped at 20GB, and features
# (~11GB) plus checkpoints (~7GB) would leave no headroom there.
!CUDA_VISIBLE_DEVICES=0 python scripts/train_ctc.py \
    --output-dir /kaggle/working/ctc-v1 \
    --cache-dir /kaggle/temp/features \
    --epochs 3 --batch-size 4 --grad-accum 8 --lr 5e-5 \
    --num-proc 4 --valid-frac 0.06 --seed 42

**If loss goes `nan` and stays there:** that's fp16 underflow in the CTC loss, not
a data problem. T4 can't do bf16, so the fixes are to lower the learning rate to
`3e-5`, or raise warmup to `--warmup-ratio 0.2`. If it persists, training in fp32
works but roughly halves throughput.

**If it OOMs:** drop to `--batch-size 2 --grad-accum 16`.

## 4b. Where is the time going?

Run this **with training stopped** — it needs the GPU to itself. It times
forward+backward on synthetic tensors and, if a feature cache exists, the real
dataloader, then says which one dominates.

A faster GPU only helps the first number. If the dataloader dominates, renting an
A100 buys nothing.

In [ ]:
!python scripts/bench.py --batch-size 4 --grad-accum 8 \
    --cache-dir /kaggle/temp/features

## 5. Validation

The honest number. `combined` is the competition metric on **held-out speakers**;
the per-language breakdown shows which language is dragging — Luganda has the
least data (5,455 rows vs ~14k each for the others), so expect it to lag.

In [ ]:
import json, pathlib
state = json.loads(pathlib.Path("/kaggle/working/ctc-v1/best/trainer_state.json").read_text()) \
    if pathlib.Path("/kaggle/working/ctc-v1/best/trainer_state.json").exists() else None
if state:
    rows = [h for h in state["log_history"] if "eval_combined" in h]
    for h in rows[-5:]:
        per = {k.replace("eval_combined_", ""): round(v, 4)
               for k, v in h.items() if k.startswith("eval_combined_")}
        print(f"step {h['step']:>6}  combined {h['eval_combined']:.4f}  "
              f"wer {h['eval_wer']:.4f}  cer {h['eval_cer']:.4f}  {per}")
else:
    print("no trainer_state.json — run training first")

## 6. Submission

Phase 1 predictions — for format validation and pipeline confidence only. The
leaderboard score it returns is not meaningful, since others may be submitting
lookups against the public labels.

When Phase 2 lands (~26 July), switch to `--phase 2 --phase2-dir <path>`.

In [ ]:
!python scripts/infer.py \
    --model /kaggle/working/ctc-v1/best \
    --phase 1 \
    --sample-submission /kaggle/input/waxal-csvs/SampleSubmission.csv \
    --out /kaggle/working/submission.csv \
    --batch-size 8

In [ ]:
import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv", escapechar="\\")
sample = pd.read_csv("/kaggle/input/waxal-csvs/SampleSubmission.csv", escapechar="\\")
assert list(sub.columns) == ["ID", "Target"], sub.columns
assert len(sub) == len(sample), (len(sub), len(sample))
assert sub.ID.tolist() == sample.ID.tolist(), "ID order must match SampleSubmission"
print(f"submission valid — {len(sub):,} rows")
sub.head()